In [6]:
API_key="API_KEY"

In [ ]:
import os
import json
import re
from google import genai
from google.genai import types

# Setup Gemini client and model
client = genai.Client(api_key=API_key)
MODEL_NAME = "models/gemini-2.0-flash"
print(f"Using model: {MODEL_NAME} for API generation")

# Instruction template
INSTRUCTIONS = (
    "Je krijgt een JSON-array met documenten met betrekking tot de coronacrisis in het Nederlands of Engels.\n"
    "Geef de resultaten terug in het Nederlands. Voor elk document extraheer je de volgende informatie:\n"
    "1. organisaties: alle namen van organisaties en instituten die in het document genoemd worden en een rol spelen in de coronacrisis of de besluitvorming eromheen. Dit kunnen overheidsinstanties, ministeries, ziekenhuizen, onderzoeksinstituten, belangenorganisaties, bedrijven, etc. zijn.\n"
    "**BELANGRIJK: Onder een organisatie of instituut verstaan we een formele entiteit met een duidelijke naam en een bepaalde structuur of functie. Voorbeelden zijn 'Ministerie van Volksgezondheid, Welzijn en Sport', 'Rijksinstituut voor Volksgezondheid en Milieu', 'Wereldgezondheidsorganisatie', 'Het Rode Kruis', etc.**\n"
    "**UITSLUITING: Extraheer GEEN websites, URL's, domeinnamen, e-mailadressen of andere online identifiers. Sluit patronen uit zoals '.nl', '.com', '.org', 'www.', '@', etc.**\n"
    "**EXTRACTIE:** Zoek eerst naar namen van organisaties/instituten die LETTERLIJK in het document voorkomen. De namen moeten minimaal 2 karakters lang zijn.\n"
    "**NORMALISATIE NAAR AFKORTINGEN:** Indien een organisatie/instituut zowel met de volledige naam als met een bekende afkorting in het document voorkomt, normaliseer de naam dan naar de afkorting. Bijvoorbeeld, als het document zowel 'Rijksinstituut voor Volksgezondheid en Milieu' als 'RIVM' noemt, gebruik dan 'RIVM'. Als alleen de volledige naam voorkomt, gebruik dan de volledige naam. Als alleen de afkorting voorkomt, gebruik dan de afkorting.\n"
    "**PRIORITEIT AAN AFKORTINGEN: Als zowel de volledige naam als de afkorting in het document staan, geef dan de voorkeur aan de afkorting in de output.**\n"
    "**EXTREEM BELANGRIJK: Genereer ABSOLUUT GEEN namen van organisaties/instituten of afkortingen die NIET in het document voorkomen. De normalisatie mag NOOIT leiden tot het toevoegen van namen of afkortingen die niet al op de een of andere manier in het document genoemd worden. Indien er geen organisaties/instituten in het document genoemd worden, laat de \"organisaties\" array dan leeg.**\n"
    "Stuur als output één JSON-object met een \"results\"-array. De \"results\"-array bevat objecten met een \"document_id\" en een \"organisaties\" array. De \"organisaties\" array bevat een lijst van strings, waarbij elke string een (genormaliseerde) naam van een organisatie/instituut is.\n"
    "Neem per document het document_id mee in de response.\n"
    "Lever uitsluitend geldige JSON zonder extra markdown‑fences, zonder trailing commas, met alle strings correct geescaped.\n"
    "Hier zijn een paar voorbeelden:\n"
    "INPUT DOCUMENT 1: 'Het RIVM heeft nieuwe richtlijnen gepubliceerd. Meer informatie is te vinden op www.rivm.nl.'\n"
    "INPUT DOCUMENT 2: 'De Wereldgezondheidsorganisatie (WHO) adviseert mondkapjes. Contact: info@who.int.'\n"
    "INPUT DOCUMENT 3: 'Het ministerie van VWS is verantwoordelijk voor het beleid.'\n"
    "INPUT DOCUMENT 4: 'Dit document beschrijft de algemene situatie rondom corona.'\n"
    "INPUT DOCUMENT 5: 'Het RIVM, oftewel het Rijksinstituut voor Volksgezondheid en Milieu, speelt een belangrijke rol.'\n"
    "INPUT DOCUMENT 6: 'Het Rijksinstituut voor Volksgezondheid en Milieu heeft nieuwe richtlijnen gepubliceerd.'\n"
    "OUTPUT 1: {\"results\": [{\"document_id\": \"voorbeeld-1\", \"organisaties\": [\"RIVM\"]}]}\n"
    "OUTPUT 2: {\"results\": [{\"document_id\": \"voorbeeld-2\", \"organisaties\": [\"Wereldgezondheidsorganisatie\", \"WHO\"]}]}\n"
    "OUTPUT 3: {\"results\": [{\"document_id\": \"voorbeeld-3\", \"organisaties\": [\"ministerie van VWS\"]}]}\n"
    "OUTPUT 4: {\"results\": [{\"document_id\": \"voorbeeld-4\", \"organisaties\": []}]}\n"
    "OUTPUT 5: {\"results\": [{\"document_id\": \"voorbeeld-5\", \"organisaties\": [\"RIVM\"]}]}\n"
    "OUTPUT 6: {\"results\": [{\"document_id\": \"voorbeeld-6\", \"organisaties\": [\"Rijksinstituut voor Volksgezondheid en Milieu\"]}]}\n"
)

# Batch directory and range
BATCH_DIR = "/home/nena-meijer/PyCharmMiscProject/information_extraction/batches"
BATCH_START = 2501
BATCH_END = 3813  # inclusive

aggregated_results = []

for i in range(BATCH_START, BATCH_END + 1):
    batch_path = os.path.join(BATCH_DIR, f"batch_{i}.json")
    if not os.path.isfile(batch_path):
        print(f"Batch {i}: bestand niet gevonden, overslaan...")
        continue

    with open(batch_path, 'r', encoding='utf-8') as f:
        try:
            batch = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Batch {i}: JSON decode error bij het inlezen van bestand: {e}")
            aggregated_results.append({'batch': i, 'error': f'File load error: {e}', 'raw_response': None})
            continue

    prompt = INSTRUCTIONS + json.dumps({'documents': batch}, ensure_ascii=False, indent=2)
    print(len(prompt))
    print(f"Processing batch {i} with {len(batch)} documents...")

    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(response_mime_type="text/plain")

    try:
        response_text = ''.join(
            chunk.text for chunk in client.models.generate_content_stream(
                model=MODEL_NAME, contents=contents, config=config
            )
        )

        # Clean up potential Markdown code fences
        cleaned = response_text.strip()
        cleaned = re.sub(r"^```json", "", cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
        cleaned = cleaned.strip()

        data = json.loads(cleaned)
        results = data.get('results', [])
        print(f"Batch {i}: parsed {len(results)} results")
        aggregated_results.extend(results)

    except json.JSONDecodeError as e:
        print(f"Batch {i} JSON parse error: {e}\nIncluding raw response in results.json")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': cleaned})

    except Exception as e:
        print(f"Batch {i}: onverwachte fout: {e}")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': None})

# Save final aggregated results
output_path = 'results_orgs_2501_tm_3813.json'
with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump({'results': aggregated_results}, outfile, ensure_ascii=False, indent=2)

print(f"Saved aggregated results: {len(aggregated_results)} entries to '{output_path}'")


In [1]:
import json
import csv
import re

# Pad naar je JSON-bestand
json_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/orgs/orgs_all.json'
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization.csv'

# JSON inladen
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

valid_results = []
extra_results = []
failed_items = []

import re

for idx, item in enumerate(data['results']):
    if 'document_id' in item:
        valid_results.append(item)
    elif 'raw_response' in item and item['raw_response']:
        raw = item['raw_response']
        try:
            # Fixes:
            raw_fixed = re.sub(r',\s*}', '}', raw)
            raw_fixed = re.sub(r'}\s*{', '},{', raw_fixed)
            raw_fixed = raw_fixed.replace('"GGD"en"', '"GGDen"')  # <-- deze regel toegevoegd
            if '"results"' not in raw_fixed:
                raw_fixed = '{"results": [' + raw_fixed + ']}'
            parsed_raw = json.loads(raw_fixed)
            extra_results.extend(parsed_raw['results'])
        except Exception as e:
            failed_items.append({'index': idx, 'batch': item.get('batch'), 'error': str(e)})
            print(f"Kon raw_response niet parsen bij item index {idx}, batch {item.get('batch')}: {e}")


# Combineer de correcte results
all_results = valid_results + extra_results

# Schrijf naar CSV
with open(csv_path, 'w', encoding='utf-8', newline='') as f_csv:
    writer = csv.writer(f_csv)
    writer.writerow(['document_id', 'name'])  # header

    for item in all_results:
        document_id = item['document_id']
        personen = item.get('organisaties', [])
        if personen:
            for persoon in personen:
                writer.writerow([document_id, persoon])
        else:
            writer.writerow([document_id, ''])  # leeg als er geen personen zijn

# Foutresultaat tonen
if failed_items:
    print(f"\nAantal niet-geparste items: {len(failed_items)}")
    for fail in failed_items:
        print(f"Index: {fail['index']}, Batch: {fail['batch']}, Fout: {fail['error']}")
else:
    print("\nAlle raw_responses succesvol geparsed.")

print(f"\nAantal geldige items: {len(all_results)}")
print(f"CSV opgeslagen als: {csv_path}")



Alle raw_responses succesvol geparsed.

Aantal geldige items: 22879
CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/Organization.csv


In [2]:
import csv
from collections import Counter

csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization.csv'

namen = []

# Lees de CSV en verzamel namen
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:  # alleen niet-lege namen
            namen.append(name)

# Tel de namen
naam_tellingen = Counter(namen)

# Sorteer op count (aflopend)
gesorteerd = naam_tellingen.most_common()

# Print de resultaten
print("Unieke namen en hun aantallen (gesorteerd):")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"Totaal unieke namen: {len(naam_tellingen)}")


Unieke namen en hun aantallen (gesorteerd):
RIVM: 9498
VWS: 4204
ministerie van volksgezondheid, welzijn en sport: 3655
GGD: 2260
OMT: 1667
vws: 1439
WHO: 1014
ggd: 1014
rivm: 831
LCI: 693
IGJ: 653
ministerie van VWS: 637
ECDC: 604
minvws: 593
NCTV: 533
Rijksinstituut voor Volksgezondheid en Milieu: 524
VNG: 496
Erasmus MC: 410
ministerie van vws: 396
Ministerie van Volksgezondheid, Welzijn en Sport: 395
NZa: 367
LCH: 365
rijksinstituut voor volksgezondheid en milieu: 353
minvws.nl: 353
ministerie van justitie en veiligheid: 352
NVWA: 331
rijksoverheid: 325
ZonMw: 303
Actiz: 294
min vws: 287
NZA: 285
omt: 276
kabinet: 269
GGD GHOR Nederland: 268
EZK: 262
ROAZ: 262
Nivel: 255
SZW: 240
OCW: 239
BZK: 237
VGN: 234
nza: 229
ZN: 227
CBS: 220
NVZ: 211
Verenso: 210
GGD GHOR: 209
LCDK: 202
zonmw: 199
NFU: 197
V&VN: 190
igj: 188
LNV: 184
Sanquin: 176
NOCNSF: 175
ggd'en: 171
TNO: 166
NVMM: 159
NHG: 155
LHV: 154
GGDGHOR: 152
FMS: 150
vng: 149
ministerie van Volksgezondheid, Welzijn en Sport: 148
g

In [6]:
import json
import csv
from collections import Counter

# Pad naar je mapping JSON
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/orgs/orgs_mapping.json'
# Pad naar je CSV
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)  # niet in mapping, origineel laten

# Tel de genormaliseerde namen
naam_tellingen = Counter(namen_genormaliseerd)

# Gesorteerd afdrukken
gesorteerd = naam_tellingen.most_common()

print("Genormaliseerde namen en hun aantallen:")
for naam, count in gesorteerd:
    print(f"{naam}: {count}")

print(f"\nTotaal unieke genormaliseerde namen: {len(naam_tellingen)}")


Genormaliseerde namen en hun aantallen:
Ministerie van Volksgezondheid, Welzijn en Sport: 12321
RIVM: 11256
GGD: 3322
OMT: 2037
WHO: 1014
NZa: 893
Ministerie van Justitie en Veiligheid: 735
LCI: 693
IGJ: 653
ECDC: 604
NCTV: 533
Ministerie van Binnenlandse Zaken en Koninkrijksrelaties: 496
VNG: 496
Ministerie van Sociale Zaken en Werkgelegenheid: 460
Rijksoverheid: 425
Erasmus MC: 410
Ministerie van Economische Zaken en Klimaat: 407
LCH: 365
Ministerie van Onderwijs, Cultuur en Wetenschap: 347
NVWA: 331
Ministerie van Landbouw, Natuur en Voedselkwaliteit: 323
ZonMw: 303
Actiz: 294
Ministerie van Buitenlandse Zaken: 274
kabinet: 269
GGD GHOR Nederland: 268
ROAZ: 262
Nivel: 255
VGN: 234
ZN: 227
CBS: 220
NVZ: 211
Verenso: 210
GGD GHOR: 209
Tweede Kamer: 204
LCDK: 202
Ministerie van Infrastructuur en Waterstaat: 200
zonmw: 199
NFU: 197
Ministerie van Defensie: 193
V&VN: 190
igj: 188
Sanquin: 176
NOCNSF: 175
ggd'en: 171
TNO: 166
NVMM: 159
NHG: 155
LHV: 154
GGDGHOR: 152
FMS: 150
vng: 149
ggd 

In [8]:
import json
import csv
from collections import Counter

# Paden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/orgs/orgs_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization_per_doc.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name

namen_genormaliseerd = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                namen_genormaliseerd.append(lookup[name_lower])
            else:
                namen_genormaliseerd.append(name)

# Tel en haal unieke namen
unieke_namen = sorted(set(namen_genormaliseerd))

# Schrijf unieke namen naar CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.writer(f_out)
    writer.writerow(['organization_id', 'name'])  # header
    for idx, naam in enumerate(unieke_namen, start=1):
        writer.writerow([idx, naam])

print(f"Unieke namen opgeslagen in: {csv_output_path}")
print(f"Totaal unieke namen: {len(unieke_namen)}")


Unieke namen opgeslagen in: /home/nena-meijer/PyCharmMiscProject/database/Organization.csv
Totaal unieke namen: 10595


In [11]:
import json
import csv

# Paden naar je bestanden
mapping_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/orgs/orgs_mapping.json'
csv_input_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization_per_doc.csv'
csv_output_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization_normalized.csv'

# Mapping inladen
with open(mapping_path, 'r', encoding='utf-8') as f_map:
    naam_mapping = json.load(f_map)

# Mapping omzetten naar dictionary voor snelle lookup
lookup = {}
for group in naam_mapping:
    main_name = list(group.keys())[0]
    for variant in group[main_name]:
        lookup[variant.lower()] = main_name  # case-insensitive mapping

# Nieuwe lijst voor rijen met genormaliseerde namen
genormaliseerde_rijen = []

# Lees CSV en pas mapping toe
with open(csv_input_path, 'r', encoding='utf-8') as f_csv:
    reader = csv.DictReader(f_csv)
    for row in reader:
        name = row['name'].strip()
        if name:
            name_lower = name.lower()
            if name_lower in lookup:
                row['name'] = lookup[name_lower]  # vervang met genormaliseerde naam
            else:
                row['name'] = name  # geen match, laat origineel staan
        else:
            row['name'] = ''  # lege waarde blijft leeg
        genormaliseerde_rijen.append(row)

# Schrijf het resultaat naar een nieuwe CSV
with open(csv_output_path, 'w', encoding='utf-8', newline='') as f_out:
    fieldnames = ['document_id', 'name']
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(genormaliseerde_rijen)

print(f"Genormaliseerde CSV opgeslagen als: {csv_output_path}")


Genormaliseerde CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/Organization_normalized.csv


In [12]:
import csv

# Paden naar je CSV's
person_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization_normalized.csv'
unique_persons_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Organization.csv'
output_csv_path = '/home/nena-meijer/PyCharmMiscProject/database/DocumentOrganization.csv'

# Stap 1: Laad person_id -> name mapping
person_mapping = {}
with open(unique_persons_csv_path, 'r', encoding='utf-8') as f_unique:
    reader = csv.DictReader(f_unique)
    for row in reader:
        person_mapping[row['name'].strip()] = row['organization_id']

# Stap 2: Verwerk de document_id, name CSV
document_person_rows = []

with open(person_csv_path, 'r', encoding='utf-8') as f_persons:
    reader = csv.DictReader(f_persons)
    for row in reader:
        document_id = row['document_id']
        name = row['name'].strip()
        if name and name in person_mapping:
            person_id = person_mapping[name]
            document_person_rows.append({'document_id': document_id, 'organization_id': person_id})
        elif name == '':
            # Optioneel: sla lege namen over, of voeg document_id met lege person_id toe
            continue
        else:
            # Naam niet gevonden in mapping, optioneel loggen
            print(f"Naam niet gevonden: {name}")

# Stap 3: Schrijf nieuwe CSV met document_id, person_id
with open(output_csv_path, 'w', encoding='utf-8', newline='') as f_out:
    writer = csv.DictWriter(f_out, fieldnames=['document_id', 'organization_id'])
    writer.writeheader()
    writer.writerows(document_person_rows)

print(f"Document-Person CSV opgeslagen als: {output_csv_path}")
print(f"Totaal koppelingen: {len(document_person_rows)}")


Document-Person CSV opgeslagen als: /home/nena-meijer/PyCharmMiscProject/database/DocumentOrganization.csv
Totaal koppelingen: 80923
